# Quality Control for the processed parquet

In [ ]:
import pandas as pd
from pathlib import Path

root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

panel = pd.read_parquet(root / "data/processed_data/analysis_panel.parquet")
print(panel.shape)
print(panel.dtypes)
panel.head(20)

In [ ]:
"""
Quality control script for analysis_panel.parquet
Tests each processing stage from processor.py and indicators.py
"""

import pandas as pd
import numpy as np
from pathlib import Path

PANEL_PATH = "/Users/francosebastiani/Documents/GitHub/Economic_Observatory/data/processed_data/analysis_panel.parquet"

df = pd.read_parquet(PANEL_PATH)

EXPECTED_SECTORS = [
    "Advanced Manufacturing",
    "Creative Industries",
    "Defence",
    "Digital and Technologies",
    "Financial Services",
    "Life Sciences",
    "Professional and Business Services",
]

INDICATOR_COLS = [
    "lq_emp", "lq_bus",
    "emp_share",
    "growth_emp", "cagr_emp",
    "growth_bus", "cagr_bus",
    "related_variety",
    "size_large_share", "size_micro_share",
]

YEARS_EMP  = list(range(2016, 2023))
YEARS_BUS  = list(range(2016, 2023))

sep = lambda title: print(f"\n{'='*60}\n{title}\n{'='*60}")

# =============================================================================
# 1. BASIC SHAPE
# =============================================================================
sep("1. BASIC SHAPE")
print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumns:\n{list(df.columns)}")

# =============================================================================
# 2. NO TOTAL ROWS (should have been removed by build_indicators)
# =============================================================================
sep("2. NO 'Total' SECTOR ROWS")
total_rows = df[df["IS8_SECTOR"] == "Total"]
print(f"'Total' rows remaining: {len(total_rows)}  (expected 0)")

# =============================================================================
# 3. SECTOR COVERAGE
# =============================================================================
sep("3. SECTOR COVERAGE")
found = sorted(df["IS8_SECTOR"].unique())
print(f"Sectors found ({len(found)}):\n  {found}")
missing = [s for s in EXPECTED_SECTORS if s not in found]
extra   = [s for s in found if s not in EXPECTED_SECTORS]
if missing: print(f"\n⚠️  Missing sectors: {missing}")
if extra:   print(f"⚠️  Unexpected sectors: {extra}")
if not missing and not extra: print(f"✓ All {len(EXPECTED_SECTORS)} IS8 sectors present, no extras")

# =============================================================================
# 4. LAD COUNT
# =============================================================================
sep("4. LAD COUNT")
n_lads = df["GEOGRAPHY_CODE"].nunique()
print(f"Unique LADs: {n_lads}  (expect 350: 296 England + 32 Scotland + 22 Wales)")

# =============================================================================
# 5. YEAR COVERAGE
# =============================================================================
sep("5. YEAR COVERAGE")
years = sorted(df["YEAR"].unique())
print(f"Years present: {years}")

# =============================================================================
# 6. PANEL BALANCE — every LAD x sector x year should exist
# =============================================================================
sep("6. PANEL BALANCE")
lads    = df["GEOGRAPHY_CODE"].unique()
sectors = df[df["IS8_SECTOR"] != "Total"]["IS8_SECTOR"].unique()
expected_rows = len(lads) * len(sectors) * len(years)
print(f"Actual rows:   {len(df):,}")
print(f"Expected rows: {expected_rows:,}  ({len(lads)} LADs × {len(sectors)} sectors × {len(years)} years)")
if len(df) != expected_rows:
    # identify gaps
    full_idx = pd.MultiIndex.from_product(
        [lads, sectors, years],
        names=["GEOGRAPHY_CODE", "IS8_SECTOR", "YEAR"]
    )
    actual_idx = pd.MultiIndex.from_frame(df[["GEOGRAPHY_CODE", "IS8_SECTOR", "YEAR"]])
    missing_combos = full_idx.difference(actual_idx)
    print(f"⚠️  Missing combinations: {len(missing_combos):,}")
    print(missing_combos.to_frame(index=False).head(20))
else:
    print("✓ Panel is balanced")

# =============================================================================
# 7. INDICATOR COLUMNS — presence and null rates
# =============================================================================
sep("7. INDICATOR NULL RATES")
for col in INDICATOR_COLS:
    if col not in df.columns:
        print(f"⚠️  MISSING column: {col}")
    else:
        null_pct = df[col].isna().mean() * 100
        flag = "⚠️ " if null_pct > 5 else "✓ "
        print(f"{flag}{col:25s}  {null_pct:5.1f}% null")

# =============================================================================
# 8. LOCATION QUOTIENT SANITY
# =============================================================================
sep("8. LOCATION QUOTIENT SANITY")
for col in ["lq_emp", "lq_bus"]:
    if col not in df.columns:
        continue
    neg = (df[col] < 0).sum()
    extreme = (df[col] > 20).sum()
    median  = df[col].median()
    print(f"{col}: median={median:.3f}  negative={neg}  >20={extreme}")
    if neg > 0:
        print(f"  ⚠️  {neg} negative LQ values — check zero-division or bad merge")

# =============================================================================
# 9. GROWTH RATE SANITY
# =============================================================================
sep("9. GROWTH RATE SANITY")
for col in ["growth_emp", "growth_bus", "cagr_emp", "cagr_bus"]:
    if col not in df.columns:
        continue
    q = df[col].quantile([0.01, 0.25, 0.5, 0.75, 0.99]).round(3).to_dict()
    print(f"{col}: p1={q[0.01]}  p25={q[0.25]}  p50={q[0.5]}  p75={q[0.75]}  p99={q[0.99]}")

# Check growth rates are only on latest year rows (they're constant per LAD x sector)
if "cagr_emp" in df.columns:
    cagr_non_null_years = df[df["cagr_emp"].notna()]["YEAR"].value_counts()
    print(f"\ncagr_emp non-null row count by year (should be same every year — it's time-invariant):")
    print(cagr_non_null_years.sort_index())

# =============================================================================
# 10. RELATED VARIETY — entropy should be >= 0
# =============================================================================
sep("10. RELATED VARIETY SANITY")
if "related_variety" in df.columns:
    neg = (df["related_variety"] < 0).sum()
    print(f"Negative entropy values: {neg}  (expected 0)")
    print(df["related_variety"].describe().round(4))

# =============================================================================
# 11. SIZE SHARES — must sum to <= 1, no negatives
# =============================================================================
sep("11. SIZE SHARE SANITY")
for col in ["size_large_share", "size_micro_share"]:
    if col not in df.columns:
        continue
    neg   = (df[col] < 0).sum()
    above = (df[col] > 1).sum()
    print(f"{col}: negative={neg}  >1={above}  median={df[col].median():.3f}")

if "size_large_share" in df.columns and "size_micro_share" in df.columns:
    combined = df["size_large_share"].fillna(0) + df["size_micro_share"].fillna(0)
    over = (combined > 1.01).sum()
    print(f"large_share + micro_share > 1: {over} rows  (expected 0)")

# =============================================================================
# 12. ONS INDICATORS — check key columns landed
# =============================================================================
sep("12. ONS INDICATOR COLUMNS")
ons_cols = [c for c in df.columns if c not in (
    ["YEAR", "GEOGRAPHY_CODE", "GEOGRAPHY_NAME", "IS8_SECTOR",
     "EMPLOYEES", "BUSINESSES"] + INDICATOR_COLS
)]
print(f"ONS columns found ({len(ons_cols)}):\n  {ons_cols}")

# Null rate per ONS col — should be consistent across years (time-invariant)
if ons_cols:
    print("\nNull rate per ONS column:")
    for col in ons_cols:
        null_pct = df[col].isna().mean() * 100
        flag = "⚠️ " if null_pct > 30 else "  "
        print(f"{flag}  {col:40s}  {null_pct:5.1f}% null")

# =============================================================================
# 13. DUPLICATE ROWS
# =============================================================================
sep("13. DUPLICATE ROWS")
key_cols = ["YEAR", "GEOGRAPHY_CODE", "IS8_SECTOR"]
dups = df.duplicated(subset=key_cols).sum()
print(f"Duplicate (YEAR, GEOGRAPHY_CODE, IS8_SECTOR) rows: {dups}  (expected 0)")
if dups > 0:
    print(df[df.duplicated(subset=key_cols, keep=False)].sort_values(key_cols).head(10))

# =============================================================================
# 14. GEOGRAPHY CODE FORMAT
# =============================================================================
sep("14. GEOGRAPHY CODE FORMAT")
bad_codes = df[~df["GEOGRAPHY_CODE"].astype(str).str.match(r"^[EWS]\d{8}$")]["GEOGRAPHY_CODE"].unique()
print(f"Malformed geography codes: {len(bad_codes)}  (expected 0)")
if len(bad_codes): print(bad_codes[:20])

# =============================================================================
# 15. SPECIFIC KNOWN RISK: working_age_pop column name
# =============================================================================
sep("15. KNOWN RISK — working_age_pop column")
wap_candidates = [c for c in df.columns if "working" in c.lower() or "age" in c.lower()]
print(f"Columns matching 'working' or 'age': {wap_candidates}")
if not wap_candidates:
    print("⚠️  No working-age population column found — business_density will be all NaN")

# =============================================================================
# SUMMARY
# =============================================================================
sep("SUMMARY")
print(f"Panel shape: {df.shape}")
print(f"LADs: {df['GEOGRAPHY_CODE'].nunique()}")
print(f"Sectors: {df['IS8_SECTOR'].nunique()}")
print(f"Years: {sorted(df['YEAR'].unique())}")
print(f"Indicators: {[c for c in INDICATOR_COLS if c in df.columns]}")
print(f"ONS cols: {len(ons_cols)}")

In [ ]:
emp = pd.read_parquet("/Users/francosebastiani/Documents/GitHub/Economic_Observatory/data/raw_data/employee_counts/Employee_counts_IS8_LADs.parquet")
print(sorted(emp["IS8_SECTOR"].unique()))

In [ ]:
crosswalk = pd.read_csv("/Users/francosebastiani/Documents/GitHub/Economic_Observatory/data/raw_data/boundaries/MSOA_(2011)_to_MSOA_(2021)_to_Local_Authority_District_(2022)_Exact_Fit_Lookup_for_EW_(V2).csv")
print(crosswalk["CHGIND"].value_counts(dropna=False))
print(crosswalk.columns.tolist())

In [ ]:
import os
boundaries = "/Users/francosebastiani/Documents/GitHub/Economic_Observatory/data/raw_data/boundaries/"
print(os.listdir(boundaries))

In [ ]:
# Check nation breakdown of your 350 LADs
df_lads = df[["GEOGRAPHY_CODE", "GEOGRAPHY_NAME"]].drop_duplicates()
df_lads["nation"] = df_lads["GEOGRAPHY_CODE"].str[0].map({"E": "England", "W": "Wales", "S": "Scotland"})
print(df_lads["nation"].value_counts())

In [ ]:
print("=== growth_emp nulls by sector ===")
print(df[df["growth_emp"].isna()].groupby("IS8_SECTOR")["GEOGRAPHY_CODE"].nunique().sort_values(ascending=False))

print("\n=== growth_bus nulls by sector ===")
print(df[df["growth_bus"].isna()].groupby("IS8_SECTOR")["GEOGRAPHY_CODE"].nunique().sort_values(ascending=False))

print("\n=== size_large_share nulls by sector ===")
print(df[df["size_large_share"].isna()].groupby("IS8_SECTOR")["GEOGRAPHY_CODE"].nunique().sort_values(ascending=False))

In [ ]:
print("=== lq_emp > 20 ===")
print(df[df["lq_emp"] > 20][["GEOGRAPHY_NAME", "IS8_SECTOR", "YEAR", "lq_emp", "EMPLOYEES", "emp_share"]]
      .sort_values("lq_emp", ascending=False)
      .to_string(index=False))